# Q1: Build Your Personalized Knowledge Base: Take your college roll number. Extract its digits. Build a pandas DataFrame with exactly 6 FAQ entries: 4 fixed entries (given in table below) and other 2 entries constructed from your own roll number digits

In [7]:
import pandas as pd

fixed_faqs = [
    {"Category": "billing", "Question": "what is the annual fee", "Answer": "The annual fee is Rs 500.", "Keywords": "fee cost price charge"},
    {"Category": "account", "Question": "how to reset password", "Answer": "Go to Settings > Reset Password.", "Keywords": "password reset login"},
    {"Category": "general", "Question": "what are your working hours", "Answer": "We are open 9 AM to 5 PM.", "Keywords": "hours timing open time"},
    {"Category": "billing", "Question": "how can i pay the fee", "Answer": "You can pay via UPI, card, or net banking.", "Keywords": "pay payment upi fee"}
]

roll_number_int = 1024160046


roll_number_str = str(roll_number_int)

last_two_digits_str = roll_number_str[-2:]
digit1 = int(last_two_digits_str[0])
digit2 = int(last_two_digits_str[1])


categories_map = ["billing", "account", "general"]

category1 = categories_map[digit1 % 3]
category2 = categories_map[digit2 % 3]

personalized_faqs = [
    {
        "Category": category1,
        "Question": f"Placeholder Q for {category1} (digit {digit1})",
        "Answer": f"Placeholder A for {category1} based on digit {digit1}. Please invent a realistic answer.",
        "Keywords": f"keyword1_{category1}, keyword2_{category1}, keyword3_{category1}"
    },
    {
        "Category": category2,
        "Question": f"Placeholder Q for {category2} (digit {digit2})",
        "Answer": f"Placeholder A for {category2} based on digit {digit2}. Please invent a realistic answer.",
        "Keywords": f"keyword1_{category2}, keyword2_{category2}, keyword3_{category2}"
    }
]

all_faqs = fixed_faqs + personalized_faqs


faq_df = pd.DataFrame(all_faqs)


print("\nYour Personalized Knowledge Base:")
print(faq_df.to_markdown(index=False))



Your Personalized Knowledge Base:
| Category   | Question                            | Answer                                                                        | Keywords                                             |
|:-----------|:------------------------------------|:------------------------------------------------------------------------------|:-----------------------------------------------------|
| billing    | what is the annual fee              | The annual fee is Rs 500.                                                     | fee cost price charge                                |
| account    | how to reset password               | Go to Settings > Reset Password.                                              | password reset login                                 |
| general    | what are your working hours         | We are open 9 AM to 5 PM.                                                     | hours timing open time                               |
| billing    | how can i 

# Q2: Generate and Score a Hypothesis - Implement a scoring function that takes a query string and returns all matching entries ranked by confidence

In [ ]:
import re
import pandas as pd

def score_faqs(query, faq_df):
    query_words = set(re.findall(r'\b\w+\b', query.lower()))

    scored_df = faq_df.copy()

    def calculate_score(row):
        score = 0
        text_to_score = f"{row['Question']} {row['Answer']} {row['Keywords']}".lower()
        question_text = row['Question'].lower()

        for word in query_words:
            if word in text_to_score:
                score += 1
            if word in question_text:
                score += 2
        return score

    scored_df['Confidence'] = scored_df.apply(calculate_score, axis=1)

    ranked_faqs_df = scored_df[scored_df['Confidence'] > 0].sort_values(by='Confidence', ascending=False).reset_index(drop=True)

    ranked_results = []
    for index, row in ranked_faqs_df.iterrows():
        ranked_results.append({
            'FAQ': row.drop('Confidence').to_dict(),
            'Confidence': row['Confidence']
        })

    return ranked_results

# Q3: Write a function same_category(category_name, df) that returns all questions belonging to a given category. Call it using the category of one of the personalized entries from Q1, and print the result.

In [ ]:
query = "how do I pay my fee"
ranked_results = score_faqs(query, faq_df)

if ranked_results:
    print(f"Ranked FAQs for query: '{query}'\n")

    flattened_results = []
    for item in ranked_results:
        faq_data = item['FAQ']
        faq_data['Confidence'] = item['Confidence']
        flattened_results.append(faq_data)

    ranked_df = pd.DataFrame(flattened_results)

    cols = ['Confidence'] + [col for col in ranked_df.columns if col != 'Confidence']
    ranked_df = ranked_df[cols]

    print(ranked_df.to_markdown(index=False))

else:
    print(f"No matching FAQs found for query: '{query}'")

Ranked FAQs for query: 'how do I pay my fee'

|   Confidence | Category   | Question                            | Answer                                                                        | Keywords                                             |
|-------------:|:-----------|:------------------------------------|:------------------------------------------------------------------------------|:-----------------------------------------------------|
|           12 | billing    | how can i pay the fee               | You can pay via UPI, card, or net banking.                                    | pay payment upi fee                                  |
|            6 | billing    | what is the annual fee              | The annual fee is Rs 500.                                                     | fee cost price charge                                |
|            4 | account    | how to reset password               | Go to Settings > Reset Password.                                          

## Q4: Pick any one entry in your knowledge base. Ask the user to input a new keyword, add it to that entry's keywords, and save your entire updated DataFrame to a CSV file named <your_roll_number>_faq_data.csv.

In [ ]:
def same_category(category_name, df):
    return df[df['Category'].str.lower() == category_name.lower()]

personalized_category_to_search = category1

category_df = same_category(personalized_category_to_search, faq_df)

print(f"\nQuestions in the '{personalized_category_to_search}' category:")

if not category_df.empty:
    print(category_df[['Question']].to_markdown(index=False))
else:
    print("No questions found for this category.")


Questions in the 'account' category:
| Question                            |
|:------------------------------------|
| how to reset password               |
| Placeholder Q for account (digit 4) |


## Q5-Using groupby, print how many FAQ entries you have per category.

In [ ]:
print("\nYour current Knowledge Base (with index for selection):")
print(faq_df.to_markdown(index=True))

while True:
    try:
        selected_index_str = input("Enter the index of the FAQ entry you want to update (e.g., 0, 1, 2): ")
        selected_index = int(selected_index_str)
        if 0 <= selected_index < len(faq_df):
            break
        else:
            print("Invalid index. Please enter a number within the displayed range.")
    except ValueError:
        print("Invalid input. Please enter a number.")

new_keyword = input(f"Enter a new keyword for FAQ at index {selected_index} (e.g., 'new_info'): ")

if new_keyword.strip():
    current_keywords = faq_df.loc[selected_index, 'Keywords']
    updated_keywords = current_keywords.strip()
    if updated_keywords and not updated_keywords.endswith(','):
        updated_keywords += ', '
    updated_keywords += new_keyword.strip()
    faq_df.loc[selected_index, 'Keywords'] = updated_keywords
    print(f"Keyword '{new_keyword}' added to FAQ entry at index {selected_index}.")
else:
    print("No new keyword provided, entry not updated.")

print("\nUpdated Knowledge Base Preview:")
print(faq_df.loc[[selected_index]].to_markdown(index=True))

csv_filename = f"{roll_number_int}_faq_data.csv"
faq_df.to_csv(csv_filename, index=False)
print(f"\nEntire updated Knowledge Base saved to '{csv_filename}'.")



Your current Knowledge Base (with index for selection):
|    | Category   | Question                            | Answer                                                                        | Keywords                                             |
|---:|:-----------|:------------------------------------|:------------------------------------------------------------------------------|:-----------------------------------------------------|
|  0 | billing    | what is the annual fee              | The annual fee is Rs 500.                                                     | fee cost price charge                                |
|  1 | account    | how to reset password               | Go to Settings > Reset Password.                                              | password reset login, share                          |
|  2 | general    | what are your working hours         | We are open 9 AM to 5 PM.                                                     | hours timing open time           

## Q6: Modify your Q2 scoring function so that if two or more entries tie for the highest score, it does not silently pick one — it prints all matching entries instead, so the user can see every equally good match. Demonstrate with one query that produces a tie (e.g. a query matching both "fee" entries) and one that doesn't.

In [6]:
import pandas as pd
import re

# --- Start of necessary data initialization for faq_df ---
# This section is included to ensure faq_df is defined even if the kernel state was reset
# or previous cells defining these variables were not executed.

fixed_faqs = [
    {"Category": "billing", "Question": "what is the annual fee", "Answer": "The annual fee is Rs 500.", "Keywords": "fee cost price charge"},
    {"Category": "account", "Question": "how to reset password", "Answer": "Go to Settings > Reset Password.", "Keywords": "password reset login"},
    {"Category": "general", "Question": "what are your working hours", "Answer": "We are open 9 AM to 5 PM.", "Keywords": "hours timing open time"},
    {"Category": "billing", "Question": "how can i pay the fee", "Answer": "You can pay via UPI, card, or net banking.", "Keywords": "pay payment upi fee"}
]

roll_number_int = 1024160046 # Assuming a fixed value for demonstration
roll_number_str = str(roll_number_int)
last_two_digits_str = roll_number_str[-2:]
digit1 = int(last_two_digits_str[0])
digit2 = int(last_two_digits_str[1])

categories_map = ["billing", "account", "general"]

category1 = categories_map[digit1 % 3]
category2 = categories_map[digit2 % 3]

personalized_faqs = [
    {
        "Category": category1,
        "Question": f"Placeholder Q for {category1} (digit {digit1})",
        "Answer": f"Placeholder A for {category1} based on digit {digit1}. Please invent a realistic answer.",
        "Keywords": f"keyword1_{category1}, keyword2_{category1}, keyword3_{category1}"
    },
    {
        "Category": category2,
        "Question": f"Placeholder Q for {category2} (digit {digit2})",
        "Answer": f"Placeholder A for {category2} based on digit {digit2}. Please invent a realistic answer.",
        "Keywords": f"keyword1_{category2}, keyword2_{category2}, keyword3_{category2}"
    }
]

all_faqs = fixed_faqs + personalized_faqs
faq_df = pd.DataFrame(all_faqs)

# --- End of necessary data initialization for faq_df ---

print("\nFAQ entries per category:")
print(faq_df.groupby('Category').size().reset_index(name='Count').to_markdown(index=False))

# --- Modified Q2 Scoring Function to handle ties for the highest score ---
def score_faqs_modified(query, faq_df):
    # FIX: Corrected regex for word boundaries from r'\\b\\w+\\b' to r'\b\w+\b'
    query_words = set(re.findall(r'\b\w+\b', query.lower()))
    scored_df = faq_df.copy()

    def calculate_score(row):
        score = 0
        text_to_score = f"{row['Question']} {row['Answer']} {row['Keywords']}".lower()
        question_text = row['Question'].lower()

        for word in query_words:
            if word in text_to_score:
                score += 1
            if word in question_text:
                score += 2
        return score

    scored_df['Confidence'] = scored_df.apply(calculate_score, axis=1)

    # Filter for scores greater than 0 first
    scored_df_positive = scored_df[scored_df['Confidence'] > 0]

    if scored_df_positive.empty:
        return [] # No matching FAQs with positive confidence

    # Find the maximum confidence score among the positive scores
    max_confidence = scored_df_positive['Confidence'].max()

    # Filter for entries that have the maximum confidence score
    ranked_faqs_df = scored_df_positive[scored_df_positive['Confidence'] == max_confidence].reset_index(drop=True)

    ranked_results = []
    for index, row in ranked_faqs_df.iterrows():
        ranked_results.append({
            'FAQ': row.drop('Confidence').to_dict(),
            'Confidence': row['Confidence']
        })
    return ranked_results

# --- Demonstration of the modified scoring function ---

def print_ranked_results(query_text, results):
    print(f"\nRanked FAQs for query: '{query_text}'\n")
    if results:
        # Flatten the results for printing as a DataFrame
        flattened_results = []
        for item in results:
            faq_data = item['FAQ']
            faq_data['Confidence'] = item['Confidence']
            flattened_results.append(faq_data)

        ranked_df = pd.DataFrame(flattened_results)

        # Reorder columns to have Confidence first
        cols = ['Confidence'] + [col for col in ranked_df.columns if col != 'Confidence']
        ranked_df = ranked_df[cols]

        print(ranked_df.to_markdown(index=False))
    else:
        print("No matching FAQs found.")

# Demonstration 1: Query producing a tie for the highest score
query_tie = "what"
results_tie = score_faqs_modified(query_tie, faq_df)
print_ranked_results(query_tie, results_tie)

# Demonstration 2: Query not producing a tie for the highest score
query_no_tie = "how do I pay my fee"
results_no_tie = score_faqs_modified(query_no_tie, faq_df)
print_ranked_results(query_no_tie, results_no_tie)


FAQ entries per category:
| Category   |   Count |
|:-----------|--------:|
| account    |       2 |
| billing    |       3 |
| general    |       1 |

Ranked FAQs for query: 'what'

|   Confidence | Category   | Question                    | Answer                    | Keywords               |
|-------------:|:-----------|:----------------------------|:--------------------------|:-----------------------|
|            3 | billing    | what is the annual fee      | The annual fee is Rs 500. | fee cost price charge  |
|            3 | general    | what are your working hours | We are open 9 AM to 5 PM. | hours timing open time |

Ranked FAQs for query: 'how do I pay my fee'

|   Confidence | Category   | Question              | Answer                                     | Keywords            |
|-------------:|:-----------|:----------------------|:-------------------------------------------|:--------------------|
|           12 | billing    | how can i pay the fee | You can pay via UPI, 